# Defining metrics

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf

spark_jars = os.environ.get("SPARK_JARS", "")
jar_list = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "")
s3a_access_key = os.environ.get("S3A_ACCESS_KEY", "")
s3a_secret_key = os.environ.get("S3A_SECRET_KEY", "")
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .appName("Tazama-Dashboard")
    .master("local[1]")
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", s3a_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory", driver_memory)
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")

Spark Version: 3.4.2


In [ ]:
WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")

In [12]:
alerts_df  = spark.read.format("hudi").load(f"{WAREHOUSE_ROOT}/gold/alerts")
cases_df   = spark.read.format("hudi").load(f"{WAREHOUSE_ROOT}/gold/cases")
tasks_df   = spark.read.format("hudi").load(f"{WAREHOUSE_ROOT}/gold/tasks")
transactions_df = spark.read.format("hudi").load(f"{WAREHOUSE_ROOT}/gold/transactions")

In [13]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

def latest_hudi_snapshot(df, key_col: str):
    """
    Keep the latest record per key_col using _hoodie_commit_time as version.
    """
    w = Window.partitionBy(key_col).orderBy(F.col("_hoodie_commit_time").desc(), F.col("_hoodie_commit_seqno").desc())
    return (
        df.withColumn("_rn", F.row_number().over(w))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )

In [14]:
alerts = latest_hudi_snapshot(alerts_df, "alert_id")
cases  = latest_hudi_snapshot(cases_df,  "case_id")
tasks  = latest_hudi_snapshot(tasks_df,  "task_id")
txns   = latest_hudi_snapshot(transactions_df, "transaction_id")  


In [15]:
from datetime import date, timedelta

def resolve_date_range(period: str = None, start_date: str = None, end_date: str = None):
    """
    Returns (start_date, end_date) as strings YYYY-MM-DD.
    end_date is inclusive.
    """
    today = date.today()

    if start_date and end_date:
        return start_date, end_date

    if period is None:
        # default: last 30 days
        period = "LAST_30_DAYS"

    period = period.upper()

    if period == "LAST_7_DAYS":
        return str(today - timedelta(days=6)), str(today)
    if period == "LAST_30_DAYS":
        return str(today - timedelta(days=29)), str(today)
    if period == "LAST_90_DAYS":
        return str(today - timedelta(days=89)), str(today)
    if period == "YEAR":
        return f"{today.year}-01-01", str(today)
    if period == "ALL_HISTORY":
        return None, None

    raise ValueError(f"Unknown period: {period}")


In [16]:
def apply_date_filter(df, date_col: str, start_date: str, end_date: str):
    """
    date_col must be a DATE or string 'YYYY-MM-DD'. If it's timestamp, we convert.
    end_date is inclusive.
    """
    if start_date is None and end_date is None:
        return df

    # make a DATE column for consistent filtering
    d = df.withColumn("_d", F.to_date(F.col(date_col)))
    return d.filter((F.col("_d") >= F.lit(start_date)) & (F.col("_d") <= F.lit(end_date))).drop("_d")


In [17]:
import pyspark.sql.functions as F

def case_inventory_metrics(
    cases,
    alerts,
    start_date=None,
    end_date=None,
    open_status_predicate=None,
    closed_status_predicate=None,
    aml_predicate=None,
    fraud_predicate=None,
    fraud_aml_predicate=None
):
    """
    Returns dict of Spark DataFrames:
      - open_case_counts_by_status
      - open_case_counts_by_type (AML/Fraud/Fraud&AML/UNKNOWN)  [derived from alerts.alert_type_norm]
      - inflow_metrics
      - outflow_metrics (includes auto-close + reopen rates)
      - disposition_distribution_closed
      - drilldowns: open_cases_list, new_cases_list, closed_cases_list, reopened_cases_list
    """

    # -------------------------
    # Case Status sets (FROM YOUR ENUM)
    # -------------------------
    AUTO_CLOSED_STATUSES = ["STATUS_71_AUTOCLOSED_CONFIRMED", "STATUS_72_AUTOCLOSED_REFUTED"]
    CLOSED_STATUSES = [
        "STATUS_81_CLOSED_REFUTED",
        "STATUS_82_CLOSED_CONFIRMED",
        "STATUS_83_CLOSED_INCONCLUSIVE",
        "STATUS_99_ABANDONED",
        # include autoclose statuses in "closed" bucket for inventory/outflow
        *AUTO_CLOSED_STATUSES,
    ]
    REOPEN_FLOW_STATUSES = [
        "STATUS_31_PENDING_CASE_REOPENING_APPROVAL"
    ]

    # -------------------------
    # Defaults that match YOUR enum (overrideable via args)
    # -------------------------
    if open_status_predicate is None:
        open_status_predicate = lambda c: ~c.isin(CLOSED_STATUSES)
    if closed_status_predicate is None:
        closed_status_predicate = lambda c: c.isin(CLOSED_STATUSES)

    # These predicates are applied to alerts.alert_type_norm (NOT cases.case_type)
    if aml_predicate is None:
        aml_predicate = lambda c: (F.upper(c) == F.lit("AML"))
    if fraud_predicate is None:
        fraud_predicate = lambda c: (F.upper(c) == F.lit("FRAUD"))
    if fraud_aml_predicate is None:
        fraud_aml_predicate = lambda c: (F.upper(c) == F.lit("FRAUD_AND_AML"))

    # -------------------------
    # Date filters
    # -------------------------
    cases_in_period = apply_date_filter(cases, "case_created_date", start_date, end_date)
    alerts_in_period = apply_date_filter(alerts, "created_at_ts", start_date, end_date)

    # -------------------------
    # Derive case_category from alerts.alert_type_norm (priority order)
    # FRAUD_AND_AML > FRAUD > AML > UNKNOWN
    # -------------------------
    alerts_case_cat = (
        alerts
        .filter(F.col("case_id").isNotNull())
        .select("case_id", "alert_type_norm")
        .withColumn("alert_type_norm_u", F.upper(F.col("alert_type_norm")))
        .withColumn(
            "_cat_rank",
            F.when(F.col("alert_type_norm_u") == "FRAUD_AND_AML", F.lit(3))
             .when(F.col("alert_type_norm_u") == "FRAUD", F.lit(2))
             .when(F.col("alert_type_norm_u") == "AML", F.lit(1))
             .otherwise(F.lit(0))
        )
        .groupBy("case_id")
        .agg(F.max("_cat_rank").alias("_cat_rank"))
        .withColumn(
            "case_category",
            F.when(F.col("_cat_rank") == 3, F.lit("FRAUD_AND_AML"))
             .when(F.col("_cat_rank") == 2, F.lit("FRAUD"))
             .when(F.col("_cat_rank") == 1, F.lit("AML"))
             .otherwise(F.lit("UNKNOWN"))
        )
        .select("case_id", "case_category")
    )

    cases_enriched = (
        cases
        .join(alerts_case_cat, "case_id", "left")
        .withColumn("case_category", F.coalesce(F.col("case_category"), F.lit("UNKNOWN")))
    )

    cases_in_period_enriched = (
        cases_in_period
        .join(alerts_case_cat, "case_id", "left")
        .withColumn("case_category", F.coalesce(F.col("case_category"), F.lit("UNKNOWN")))
    )

    # -------------------------
    # Inventory: Open / Closed
    # -------------------------
    open_cases = cases_enriched.filter(open_status_predicate(F.col("status")))
    closed_cases = cases_enriched.filter(closed_status_predicate(F.col("status")))

    # a) Total Open Cases by status
    open_by_status = (
        open_cases.groupBy("status")
                 .agg(F.countDistinct("case_id").alias("open_cases"))
                 .orderBy(F.desc("open_cases"))
    )

    # a) Open Cases by category
    open_by_type = (
        open_cases.groupBy("case_category")
                 .agg(F.countDistinct("case_id").alias("open_cases"))
                 .orderBy(F.desc("open_cases"))
    )

    # -------------------------
    # b) New Cases (Inflow)
    # -------------------------
    # Alerts that resulted in a case during the period
    alerts_to_case = alerts_in_period.filter(F.col("case_id").isNotNull()).select("case_id").distinct()
    inflow_alert_cases = alerts_to_case.count()

    # Cases manually created during the period (non automatic system)
    inflow_manual_cases = (
        cases_in_period
        .filter(F.col("case_creation_type").isNotNull())
        .filter(F.col("case_creation_type") != F.lit("AUTOMATIC_SYSTEM"))
        .select("case_id").distinct()
        .count()
    )

    inflow_metrics = spark.createDataFrame(
        [(int(inflow_alert_cases), int(inflow_manual_cases))],
        ["alerts_resulting_in_case_during_period", "manually_created_cases_during_period"]
    )

    # -------------------------
    # c) Closed Cases (Outflow)
    # -------------------------
    closed_in_period = apply_date_filter(closed_cases, "case_updated_date", start_date, end_date)

    # Auto-closed by ATM during period
    auto_closed_in_period = (
        closed_in_period.filter(F.col("status").isin(AUTO_CLOSED_STATUSES))
                        .select("case_id").distinct()
    )

    # Reopened cases definition:
    # We only have current snapshot, so we can count cases currently in reopening flow.
    reopened_cases = cases_enriched.filter(F.col("status").isin(REOPEN_FLOW_STATUSES)).select("case_id").distinct()

    auto_closed_count = auto_closed_in_period.count()
    reopened_auto_closed_count = auto_closed_in_period.join(reopened_cases, "case_id", "inner").count()

    closed_count = closed_in_period.select("case_id").distinct().count()
    reopened_closed_count = closed_in_period.select("case_id").distinct().join(reopened_cases, "case_id", "inner").count()

    auto_closure_reopen_rate = (reopened_auto_closed_count / auto_closed_count) if auto_closed_count else 0.0
    reopened_case_rate = (reopened_closed_count / closed_count) if closed_count else 0.0

    outflow_metrics = spark.createDataFrame(
        [(
            int(auto_closed_count),
            int(reopened_auto_closed_count),
            float(auto_closure_reopen_rate),
            int(closed_count),
            int(reopened_closed_count),
            float(reopened_case_rate)
        )],
        [
            "auto_closed_by_atm_during_period",
            "auto_closed_reopened_count",
            "auto_closure_reopen_rate",
            "closed_cases_during_period",
            "reopened_closed_cases_count",
            "reopened_case_rate"
        ]
    )

    # -------------------------
    # Disposition Distribution for Closed Cases (from status)
    # -------------------------
    disposition_df = (
        closed_in_period
        .withColumn(
            "disposition",
            F.when(F.col("status").isin("STATUS_71_AUTOCLOSED_CONFIRMED", "STATUS_82_CLOSED_CONFIRMED"), F.lit("CONFIRMED"))
             .when(F.col("status").isin("STATUS_72_AUTOCLOSED_REFUTED", "STATUS_81_CLOSED_REFUTED"), F.lit("REFUTED"))
             .when(F.col("status") == "STATUS_83_CLOSED_INCONCLUSIVE", F.lit("INCONCLUSIVE"))
             .when(F.col("status") == "STATUS_99_ABANDONED", F.lit("ABANDONED"))
             .otherwise(F.lit("UNKNOWN"))
        )
        .groupBy("disposition")
        .agg(F.countDistinct("case_id").alias("closed_cases"))
        .withColumn("pct_of_closed",
                    F.round(F.col("closed_cases") / F.sum("closed_cases").over(Window.partitionBy()) * 100.0, 2))
        .orderBy(F.desc("closed_cases"))
    )

    # -------------------------
    # Drilldowns
    # -------------------------
    open_cases_list = open_cases.select(
        "case_id","tenant_id","priority","status","case_creation_type",
        "case_category","case_created_ts","case_updated_ts"
    )

    new_cases_list = cases_in_period_enriched.select(
        "case_id","tenant_id","priority","status","case_creation_type",
        "case_category","case_created_ts"
    )

    closed_cases_list = closed_in_period.select(
        "case_id","tenant_id","priority","status","case_category",
        "case_updated_ts","case_updated_date"
    )

    reopened_cases_list = (
        reopened_cases.join(cases_enriched, "case_id")
                      .select("case_id","tenant_id","priority","status","case_category","case_updated_ts")
    )

    return {
        "open_case_counts_by_status": open_by_status,
        "open_case_counts_by_type": open_by_type,
        "inflow_metrics": inflow_metrics,
        "outflow_metrics": outflow_metrics,
        "disposition_distribution_closed": disposition_df,
        "open_cases_list": open_cases_list,
        "new_cases_list": new_cases_list,
        "closed_cases_list": closed_cases_list,
        "reopened_cases_list": reopened_cases_list
    }


In [18]:
def task_workload_metrics(tasks, start_date=None, end_date=None):
    tasks_in_period = apply_date_filter(tasks, "task_created_date", start_date, end_date)

    # Tasks by status
    tasks_by_status = (
        tasks_in_period.groupBy("status")
                       .agg(F.countDistinct("task_id").alias("tasks"))
                       .orderBy(F.desc("tasks"))
    )

    # Open tasks by task type (task_type may be null; fallback to task_name)
    tasks_typed = tasks_in_period.withColumn(
        "task_type_norm",
        F.when(F.col("task_type").isNotNull(), F.col("task_type")).otherwise(F.col("task_name"))
    )

    open_tasks = tasks_typed.filter(F.col("is_completed") == 0)

    open_tasks_by_type = (
        open_tasks.groupBy("task_type_norm")
                 .agg(F.countDistinct("task_id").alias("open_tasks"))
                 .orderBy(F.desc("open_tasks"))
    )

    # Average number of open tasks per user
    # Note: unassigned tasks have assigned_user_id = null; include as "UNASSIGNED"
    open_tasks_by_user = (
        open_tasks.withColumn("assignee", F.coalesce(F.col("assigned_user_id"), F.lit("UNASSIGNED")))
                 .groupBy("assignee")
                 .agg(F.countDistinct("task_id").alias("open_tasks"))
    )

    avg_open_tasks_per_user = open_tasks_by_user.agg(F.avg("open_tasks").alias("avg_open_tasks_per_user"))

    # Drilldowns
    open_tasks_list = open_tasks.select("task_id","case_id","task_name","task_type","status","assigned_user_id","task_created_ts","sla_deadline_ts","sla_breached")

    return {
        "tasks_by_status": tasks_by_status,
        "open_tasks_by_type": open_tasks_by_type,
        "avg_open_tasks_per_user": avg_open_tasks_per_user,
        "open_tasks_list": open_tasks_list,
        "open_tasks_by_user": open_tasks_by_user.orderBy(F.desc("open_tasks"))
    }


In [19]:
def investigation_metrics(tasks, cases, start_date=None, end_date=None):
    tasks_in_period = apply_date_filter(tasks, "task_created_date", start_date, end_date)

    # Investigation tasks by status: use candidate_group == 'INVESTIGATIONS' (from your sample)
    inv_tasks = tasks_in_period.filter(F.col("candidate_group") == F.lit("INVESTIGATIONS"))

    inv_tasks_by_status = (
        inv_tasks.groupBy("status")
                 .agg(F.countDistinct("task_id").alias("investigation_tasks"))
                 .orderBy(F.desc("investigation_tasks"))
    )

    # Investigations created during period = cases created during period (assuming case ~= investigation)
    cases_in_period = apply_date_filter(cases, "case_created_date", start_date, end_date)
    investigations_created = cases_in_period.count()

    # Investigations closed during period (needs closed predicate – placeholder using "CLOSED")
    closed_cases_in_period = apply_date_filter(
        cases.filter(F.col("status").contains("CLOSED")),
        "case_updated_date",
        start_date,
        end_date
    )
    investigations_closed = closed_cases_in_period.count()

    inv_flow = spark.createDataFrame(
        [(int(investigations_created), int(investigations_closed))],
        ["investigations_created_during_period", "investigations_closed_during_period"]
    )

    # SAR/STR filings during period: not explicitly present; heuristic using task_name contains SAR/STR
    sar_str = inv_tasks.filter(
        F.upper(F.col("task_name")).contains("SAR") | F.upper(F.col("task_name")).contains("STR")
    )
    sar_str_count = sar_str.count()
    sar_str_metric = spark.createDataFrame([(int(sar_str_count),)], ["sar_str_filings_during_period"])

    # Drilldowns
    inv_tasks_list = inv_tasks.select("task_id","case_id","task_name","status","assigned_user_id","task_created_ts","task_completed_ts","sla_breached")

    return {
        "investigation_tasks_by_status": inv_tasks_by_status,
        "investigation_flow": inv_flow,
        "sar_str_filings": sar_str_metric,
        "investigation_tasks_list": inv_tasks_list
    }


In [20]:
def case_age_and_timeliness_metrics(cases, alerts, start_date=None, end_date=None, open_status_predicate=None):
    if open_status_predicate is None:
        open_status_predicate = lambda c: ~c.contains("CLOSED")  # fallback

    # Open cases for age buckets
    open_cases = cases.filter(open_status_predicate(F.col("status")))

    # Age in days (from case_created_ts to today)
    open_cases_age = open_cases.withColumn(
        "case_age_days",
        F.datediff(F.current_date(), F.to_date("case_created_ts"))
    )

    # Bucketize
    age_bucketed = open_cases_age.withColumn(
        "age_bucket",
        F.when(F.col("case_age_days") <= 7,  F.lit("0-7 days (New)"))
         .when(F.col("case_age_days") <= 14, F.lit("8-14 days (Active)"))
         .when(F.col("case_age_days") <= 30, F.lit("15-30 days (Maturing)"))
         .when(F.col("case_age_days") <= 60, F.lit("31-60 days (Aged)"))
         .otherwise(F.lit("60+ days (Critical backlog)"))
    )

    age_histogram = (
        age_bucketed.groupBy("age_bucket")
                    .agg(F.countDistinct("case_id").alias("open_cases"))
                    .orderBy("age_bucket")
    )

    # Drilldown: open cases with age bucket
    open_cases_age_list = age_bucketed.select("case_id","tenant_id","priority","status","case_created_ts","case_age_days","age_bucket")

    # MTTD: mean time to disposition (creation -> updated) for closed cases
    closed_cases = cases.filter(F.col("status").contains("CLOSED"))

    closed_in_period = apply_date_filter(closed_cases, "case_updated_date", start_date, end_date)

    closed_with_mttd = closed_in_period.withColumn(
        "mttd_hours",
        (F.col("created_to_updated_ms") / F.lit(1000*60*60)).cast("double")
    )

    mttd = closed_with_mttd.agg(F.avg("mttd_hours").alias("mean_time_to_disposition_hours"))

    # Alert-to-Disposition Time: join alerts->cases via case_id (alerts created in period)
    alerts_in_period = apply_date_filter(alerts.filter(F.col("case_id").isNotNull()), "created_at_ts", start_date, end_date)
    alert_case = (
        alerts_in_period.select("alert_id","case_id","created_at_ts")
        .join(cases.select("case_id","case_type","case_updated_ts","status"), "case_id", "left")
        .withColumn("alert_to_disposition_hours",
            (F.unix_timestamp("case_updated_ts") - F.unix_timestamp("created_at_ts")) / F.lit(3600.0)
        )
    )

    # By case type (Fraud/AML) requires case_type populated. Still compute grouped view.
    alert_to_disp_by_case_type = (
        alert_case.groupBy("case_type")
                 .agg(F.avg("alert_to_disposition_hours").alias("avg_alert_to_disposition_hours"),
                      F.countDistinct("case_id").alias("cases"))
                 .orderBy(F.desc("cases"))
    )

    # By disposition: not present in schema. Placeholder uses status as proxy.
    alert_to_disp_by_status = (
        alert_case.groupBy("status")
                 .agg(F.avg("alert_to_disposition_hours").alias("avg_alert_to_disposition_hours"),
                      F.countDistinct("case_id").alias("cases"))
                 .orderBy(F.desc("cases"))
    )

    return {
        "case_age_histogram_open": age_histogram,
        "open_cases_age_list": open_cases_age_list,
        "mttd": mttd,
        "alert_to_disposition_by_case_type": alert_to_disp_by_case_type,
        "alert_to_disposition_by_status_proxy": alert_to_disp_by_status,
        "alert_case_timeliness_drilldown": alert_case.select("alert_id","case_id","case_type","status","created_at_ts","case_updated_ts","alert_to_disposition_hours")
    }


In [21]:
def high_value_open_case_tracking(
    cases, alerts,
    start_date=None, end_date=None,
    open_status_predicate=None,
    value_threshold=1000.0
):
    if open_status_predicate is None:
        open_status_predicate = lambda c: ~c.contains("CLOSED")  # fallback

    open_cases = cases.filter(open_status_predicate(F.col("status"))).select("case_id","tenant_id","priority","status","case_created_ts")

    # Alerts tied to cases
    alerts_with_case = alerts.filter(F.col("case_id").isNotNull())

    # If you want to scope to date range, filter alerts by created_at_ts
    alerts_in_period = apply_date_filter(alerts_with_case, "created_at_ts", start_date, end_date)

    # Coerce amount to double
    alerts_amt = alerts_in_period.withColumn("tx_amount_num", F.col("tx_amount").cast("double"))

    case_values = (
        alerts_amt.groupBy("case_id")
                 .agg(F.sum("tx_amount_num").alias("case_value_at_risk"),
                      F.countDistinct("alert_id").alias("alerts_in_case"))
    )

    open_case_values = open_cases.join(case_values, "case_id", "left").fillna({"case_value_at_risk": 0.0, "alerts_in_case": 0})

    high_value_open = open_case_values.filter(F.col("case_value_at_risk") >= F.lit(float(value_threshold)))

    summary = high_value_open.agg(
        F.countDistinct("case_id").alias("open_cases_above_threshold"),
        F.sum("case_value_at_risk").alias("total_value_at_risk"),
        F.avg("case_value_at_risk").alias("avg_value_per_open_case")
    )

    top10 = high_value_open.orderBy(F.desc("case_value_at_risk")).limit(10)

    return {
        "high_value_summary": summary,
        "top10_open_cases_by_value": top10,
        "high_value_open_cases_list": high_value_open.orderBy(F.desc("case_value_at_risk"))
    }


In [22]:
def build_supervisor_dashboard_metrics(
    alerts, cases, tasks,
    period=None, start_date=None, end_date=None,
    value_threshold=1000.0
):
    start_date, end_date = resolve_date_range(period=period, start_date=start_date, end_date=end_date)

    # Define open/closed status logic (YOU should align these with your real status taxonomy)
    open_pred = lambda c: ~c.contains("CLOSED")
    closed_pred = lambda c: c.contains("CLOSED")

    out = {}

    out["date_range"] = (start_date, end_date)

    out.update(case_inventory_metrics(
        cases=cases,
        alerts=alerts,
        start_date=start_date,
        end_date=end_date,
        open_status_predicate=open_pred,
        closed_status_predicate=closed_pred
    ))

    out.update(task_workload_metrics(tasks=tasks, start_date=start_date, end_date=end_date))
    out.update(investigation_metrics(tasks=tasks, cases=cases, start_date=start_date, end_date=end_date))
    out.update(case_age_and_timeliness_metrics(cases=cases, alerts=alerts, start_date=start_date, end_date=end_date, open_status_predicate=open_pred))
    out.update(high_value_open_case_tracking(cases=cases, alerts=alerts, start_date=start_date, end_date=end_date, open_status_predicate=open_pred, value_threshold=value_threshold))

    return out


In [23]:
metrics = build_supervisor_dashboard_metrics(
    alerts=alerts,
    cases=cases,
    tasks=tasks,
    period="LAST_30_DAYS",
    value_threshold=5000.0
)

print("Date range:", metrics["date_range"])

print("\n=== Open Case Counts by Status ===")
metrics["open_case_counts_by_status"].show(truncate=False)

print("\n=== Inflow Metrics ===")
metrics["inflow_metrics"].show(truncate=False)

print("\n=== Outflow Metrics ===")
metrics["outflow_metrics"].show(truncate=False)

print("\n=== Tasks by Status ===")
metrics["tasks_by_status"].show(truncate=False)

print("\n=== Open Case Age Histogram ===")
metrics["case_age_histogram_open"].show(truncate=False)

print("\n=== High Value Summary ===")
metrics["high_value_summary"].show(truncate=False)

print("\n=== Top 10 Open Cases by Value ===")
metrics["top10_open_cases_by_value"].show(truncate=False)


Date range: ('2026-03-16', '2026-04-14')

=== Open Case Counts by Status ===
+---------------+----------+
|status         |open_cases|
+---------------+----------+
|STATUS_00_DRAFT|128381    |
+---------------+----------+


=== Inflow Metrics ===


+--------------------------------------+------------------------------------+
|alerts_resulting_in_case_during_period|manually_created_cases_during_period|
+--------------------------------------+------------------------------------+
|128378                                |0                                   |
+--------------------------------------+------------------------------------+


=== Outflow Metrics ===


+--------------------------------+--------------------------+------------------------+--------------------------+---------------------------+------------------+
|auto_closed_by_atm_during_period|auto_closed_reopened_count|auto_closure_reopen_rate|closed_cases_during_period|reopened_closed_cases_count|reopened_case_rate|
+--------------------------------+--------------------------+------------------------+--------------------------+---------------------------+------------------+
|0                               |0                         |0.0                     |0                         |0                          |0.0               |
+--------------------------------+--------------------------+------------------------+--------------------------+---------------------------+------------------+


=== Tasks by Status ===
+--------------------+------+
|status              |tasks |
+--------------------+------+
|STATUS_01_UNASSIGNED|128378|
+--------------------+------+


=== Open Case Age

# Visualisations

In [24]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pyspark.sql.functions as F
from pyspark.sql.types import TimestampType, DateType

def spark_small_to_pandas(df, max_rows=2000):
    """
    Safe conversion helper for KPI/aggregate outputs.
    - Casts DATE/TIMESTAMP columns to string to avoid pandas dtype issues.
    """
    # Cast timestamp/date columns to string to avoid pandas dtype precision error
    for field in df.schema.fields:
        if isinstance(field.dataType, (TimestampType, DateType)):
            df = df.withColumn(field.name, F.col(field.name).cast("string"))
    return df.limit(max_rows).toPandas()



In [25]:
def render_supervisor_kpi_cards(metrics, title="Supervisor Case Operations Dashboard"):
    start_date, end_date = metrics["date_range"]

    # ---- Pull small aggregates into pandas ----
    inflow_pd = spark_small_to_pandas(metrics["inflow_metrics"])
    outflow_pd = spark_small_to_pandas(metrics["outflow_metrics"])
    high_value_pd = spark_small_to_pandas(metrics["high_value_summary"])

    # These are distributions (small tables)
    open_by_status_pd = spark_small_to_pandas(metrics["open_case_counts_by_status"])
    open_by_type_pd   = spark_small_to_pandas(metrics["open_case_counts_by_type"])
    tasks_by_status_pd = spark_small_to_pandas(metrics["tasks_by_status"])
    age_bucket_pd = spark_small_to_pandas(metrics["case_age_histogram_open"])
    avg_open_tasks_pd = spark_small_to_pandas(metrics["avg_open_tasks_per_user"])

    # ---- Compute KPI values ----
    total_open_cases = int(open_by_status_pd["open_cases"].sum()) if len(open_by_status_pd) else 0
    total_new_cases_alert = int(inflow_pd.loc[0, "alerts_resulting_in_case_during_period"]) if len(inflow_pd) else 0
    total_new_cases_manual = int(inflow_pd.loc[0, "manually_created_cases_during_period"]) if len(inflow_pd) else 0

    closed_cases = int(outflow_pd.loc[0, "closed_cases_during_period"]) if len(outflow_pd) else 0
    auto_closed = int(outflow_pd.loc[0, "auto_closed_by_atm_during_period"]) if len(outflow_pd) else 0
    reopened_rate = float(outflow_pd.loc[0, "reopened_case_rate"]) * 100.0 if len(outflow_pd) else 0.0
    auto_reopen_rate = float(outflow_pd.loc[0, "auto_closure_reopen_rate"]) * 100.0 if len(outflow_pd) else 0.0

    total_tasks = int(tasks_by_status_pd["tasks"].sum()) if len(tasks_by_status_pd) else 0
    avg_open_tasks_per_user = float(avg_open_tasks_pd.loc[0, "avg_open_tasks_per_user"]) if len(avg_open_tasks_pd) else 0.0

    high_value_open_cases = int(high_value_pd.loc[0, "open_cases_above_threshold"]) if len(high_value_pd) else 0
    total_value_at_risk = float(high_value_pd.loc[0, "total_value_at_risk"]) if len(high_value_pd) else 0.0
    avg_value_per_case = float(high_value_pd.loc[0, "avg_value_per_open_case"]) if len(high_value_pd) else 0.0

    # ---- KPI definitions with basic thresholds ----
    # Adjust thresholds based on your operational expectations.
    kpis = [
        {"title": "Total Open Cases", "value": total_open_cases, "unit": "",
         "threshold_good": 50, "threshold_fair": 200, "higher_is_better": False},

        {"title": "New Cases (Alerts)", "value": total_new_cases_alert, "unit": "",
         "threshold_good": 50, "threshold_fair": 150, "higher_is_better": False},

        {"title": "New Cases (Manual)", "value": total_new_cases_manual, "unit": "",
         "threshold_good": 10, "threshold_fair": 50, "higher_is_better": False},

        {"title": "Closed Cases", "value": closed_cases, "unit": "",
         "threshold_good": 50, "threshold_fair": 20, "higher_is_better": True},

        {"title": "Auto-Closed (ATM)", "value": auto_closed, "unit": "",
         "threshold_good": 30, "threshold_fair": 10, "higher_is_better": True},

        {"title": "Reopened Case Rate", "value": reopened_rate, "unit": "%",
         "threshold_good": 2.0, "threshold_fair": 5.0, "higher_is_better": False},

        {"title": "Auto-Closure Reopen Rate", "value": auto_reopen_rate, "unit": "%",
         "threshold_good": 1.0, "threshold_fair": 3.0, "higher_is_better": False},

        {"title": "Total Tasks", "value": total_tasks, "unit": "",
         "threshold_good": 200, "threshold_fair": 500, "higher_is_better": False},

        {"title": "Avg Open Tasks / User", "value": avg_open_tasks_per_user, "unit": "",
         "threshold_good": 10, "threshold_fair": 20, "higher_is_better": False},

        {"title": "High-Value Open Cases", "value": high_value_open_cases, "unit": "",
         "threshold_good": 5, "threshold_fair": 15, "higher_is_better": False},

        {"title": "Total Value at Risk", "value": total_value_at_risk, "unit": "",
         "threshold_good": 1_000_000, "threshold_fair": 5_000_000, "higher_is_better": False},

        {"title": "Avg Value / Open Case", "value": avg_value_per_case, "unit": "",
         "threshold_good": 10_000, "threshold_fair": 50_000, "higher_is_better": False},
    ]

    # ---- Create 4x3 grid of indicators ----
    fig = make_subplots(
        rows=4, cols=3,
        specs=[[{'type': 'indicator'}]*3]*4,
        vertical_spacing=0.15, horizontal_spacing=0.12
    )

    for idx, kpi in enumerate(kpis):
        row = (idx // 3) + 1
        col = (idx % 3) + 1

        # Determine status color
        if kpi["higher_is_better"]:
            if kpi["value"] >= kpi["threshold_good"]:
                color = "#2ecc71"
            elif kpi["value"] >= kpi["threshold_fair"]:
                color = "#f39c12"
            else:
                color = "#e74c3c"
        else:
            if kpi["value"] <= kpi["threshold_good"]:
                color = "#2ecc71"
            elif kpi["value"] <= kpi["threshold_fair"]:
                color = "#f39c12"
            else:
                color = "#e74c3c"

        # Gauge range
        # (for money-like metrics, keep it bigger)
        gauge_max = kpi["threshold_fair"] * 1.5 if kpi["threshold_fair"] else (kpi["value"] * 1.5 + 1)

        fig.add_trace(
            go.Indicator(
                mode="number+gauge",
                value=float(kpi["value"]),
                title={"text": kpi["title"]},
                number={"suffix": kpi["unit"]},
                gauge={
                    "axis": {"range": [0, gauge_max]},
                    "bar": {"color": color},
                    "steps": [
                        {"range": [0, kpi["threshold_good"]], "color": "#e8f5e9"},
                        {"range": [kpi["threshold_good"], kpi["threshold_fair"]], "color": "#fff3cd"},
                        {"range": [kpi["threshold_fair"], gauge_max], "color": "#f8d7da"},
                    ],
                },
            ),
            row=row, col=col
        )

    fig.update_layout(
        title_text=f"<b>{title}</b><br><sub>Date Range: {start_date} to {end_date}</sub>",
        showlegend=False,
        height=1200,
        width=1400
    )

    fig.show()


In [26]:
def render_supervisor_operational_charts(metrics, title="Supervisor Operational Monitoring"):
    start_date, end_date = metrics["date_range"]

    open_by_status_pd = spark_small_to_pandas(metrics["open_case_counts_by_status"])
    open_by_type_pd   = spark_small_to_pandas(metrics["open_case_counts_by_type"])
    tasks_by_status_pd = spark_small_to_pandas(metrics["tasks_by_status"])
    age_bucket_pd = spark_small_to_pandas(metrics["case_age_histogram_open"])

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Open Cases by Status",
            "Open Cases by Category (Fraud/AML)",
            "Tasks by Status",
            "Open Case Age Distribution"
        ),
        specs=[
            [{"type": "xy"}, {"type": "domain"}],  # pie needs domain
            [{"type": "xy"}, {"type": "xy"}]
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.12
    )

    # Chart 1: Open cases by status (bar)
    fig.add_trace(
        go.Bar(
            x=open_by_status_pd["status"] if len(open_by_status_pd) else [],
            y=open_by_status_pd["open_cases"] if len(open_by_status_pd) else [],
            name="Open Cases"
        ),
        row=1, col=1
    )

    # Chart 2: Open cases by type/category (donut)
    fig.add_trace(
        go.Pie(
            labels=open_by_type_pd["case_category"] if len(open_by_type_pd) else [],
            values=open_by_type_pd["open_cases"] if len(open_by_type_pd) else [],
            hole=0.5,
            name="Case Category"
        ),
        row=1, col=2
    )

    # Chart 3: Tasks by status (bar)
    fig.add_trace(
        go.Bar(
            x=tasks_by_status_pd["status"] if len(tasks_by_status_pd) else [],
            y=tasks_by_status_pd["tasks"] if len(tasks_by_status_pd) else [],
            name="Tasks"
        ),
        row=2, col=1
    )

    # Chart 4: Age distribution (bar; buckets)
    fig.add_trace(
        go.Bar(
            x=age_bucket_pd["age_bucket"] if len(age_bucket_pd) else [],
            y=age_bucket_pd["open_cases"] if len(age_bucket_pd) else [],
            name="Open Cases"
        ),
        row=2, col=2
    )

    fig.update_layout(
        title_text=f"<b>{title}</b><br><sub>Date Range: {start_date} to {end_date}</sub>",
        height=900,
        width=1400,
        showlegend=False
    )

    fig.show()


In [27]:
def render_top10_high_value_table(metrics, title="Top 10 High-Value Open Cases"):
    start_date, end_date = metrics["date_range"]
    top10_pd = spark_small_to_pandas(metrics["top10_open_cases_by_value"], max_rows=10)

    fig = go.Figure(
        data=[
            go.Table(
                header=dict(
                    values=["case_id", "tenant_id", "priority", "status", "case_created_ts", "case_value_at_risk", "alerts_in_case"],
                    fill_color="#eeeeee",
                    align="left"
                ),
                cells=dict(
                    values=[
                        top10_pd["case_id"] if "case_id" in top10_pd else [],
                        top10_pd["tenant_id"] if "tenant_id" in top10_pd else [],
                        top10_pd["priority"] if "priority" in top10_pd else [],
                        top10_pd["status"] if "status" in top10_pd else [],
                        top10_pd["case_created_ts"] if "case_created_ts" in top10_pd else [],
                        top10_pd["case_value_at_risk"] if "case_value_at_risk" in top10_pd else [],
                        top10_pd["alerts_in_case"] if "alerts_in_case" in top10_pd else [],
                    ],
                    align="left"
                )
            )
        ]
    )

    fig.update_layout(
        title_text=f"<b>{title}</b><br><sub>Date Range: {start_date} to {end_date}</sub>",
        height=450,
        width=1400
    )
    fig.show()


In [28]:
def compute_supervisor_trends(cases, tasks, period="day", start_date=None, end_date=None):
    """
    period: 'day' | 'week' | 'month'
    Returns Spark DF with inflow/outflow/task trends.
    """

    # Date filter
    cases_f = apply_date_filter(cases, "case_created_date", start_date, end_date)
    tasks_f = apply_date_filter(tasks, "task_created_date", start_date, end_date)

    # Bucket column
    if period == "day":
        bucket = F.to_date(F.col("case_created_ts"))
        bucket_tasks = F.to_date(F.col("task_created_ts"))
    elif period == "week":
        bucket = F.date_trunc("week", F.col("case_created_ts"))
        bucket_tasks = F.date_trunc("week", F.col("task_created_ts"))
    else:
        bucket = F.date_trunc("month", F.col("case_created_ts"))
        bucket_tasks = F.date_trunc("month", F.col("task_created_ts"))

    CLOSED_STATUSES = [
        "STATUS_71_AUTOCLOSED_CONFIRMED", "STATUS_72_AUTOCLOSED_REFUTED",
        "STATUS_81_CLOSED_REFUTED", "STATUS_82_CLOSED_CONFIRMED", "STATUS_83_CLOSED_INCONCLUSIVE",
        "STATUS_99_ABANDONED"
    ]
    AUTO_CLOSED_STATUSES = ["STATUS_71_AUTOCLOSED_CONFIRMED", "STATUS_72_AUTOCLOSED_REFUTED"]

    # Inflow (cases created)
    inflow = (
        cases_f.withColumn("period", bucket)
               .groupBy("period")
               .agg(F.countDistinct("case_id").alias("cases_created"))
    )

    # Outflow (cases closed) using case_updated_ts as close time
    closed = (
        cases.filter(F.col("status").isin(CLOSED_STATUSES))
             .transform(lambda df: apply_date_filter(df, "case_updated_date", start_date, end_date))
             .withColumn("period", F.date_trunc(period, F.col("case_updated_ts")))
             .groupBy("period")
             .agg(
                 F.countDistinct("case_id").alias("cases_closed"),
                 F.countDistinct(F.when(F.col("status").isin(AUTO_CLOSED_STATUSES), F.col("case_id"))).alias("cases_autoclosed")
             )
    )

    # Tasks created + breached
    tasks_trend = (
        tasks_f.withColumn("period", bucket_tasks)
              .groupBy("period")
              .agg(
                  F.countDistinct("task_id").alias("tasks_created"),
                  F.countDistinct(F.when(F.col("sla_breached") == 1, F.col("task_id"))).alias("tasks_sla_breached")
              )
    )

    # Join trends
    trend = (
        inflow.join(closed, "period", "full")
              .join(tasks_trend, "period", "full")
              .fillna(0)
              .orderBy("period")
    )
    return trend

def render_supervisor_trends(trend_df, title="Supervisor Trend Dashboard"):
    trend_pd = spark_small_to_pandas(trend_df, max_rows=5000)
    if len(trend_pd) == 0:
        print("No trend data for selected range.")
        return

    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=("Case Inflow/Outflow", "Auto-Closures", "Task Load & SLA Breaches"),
        vertical_spacing=0.12
    )

    fig.add_trace(go.Scatter(x=trend_pd["period"], y=trend_pd["cases_created"], mode="lines+markers", name="Cases Created"), row=1, col=1)
    fig.add_trace(go.Scatter(x=trend_pd["period"], y=trend_pd["cases_closed"], mode="lines+markers", name="Cases Closed"), row=1, col=1)

    fig.add_trace(go.Scatter(x=trend_pd["period"], y=trend_pd["cases_autoclosed"], mode="lines+markers", name="Auto-Closed Cases"), row=2, col=1)

    fig.add_trace(go.Scatter(x=trend_pd["period"], y=trend_pd["tasks_created"], mode="lines+markers", name="Tasks Created"), row=3, col=1)
    fig.add_trace(go.Scatter(x=trend_pd["period"], y=trend_pd["tasks_sla_breached"], mode="lines+markers", name="SLA Breached Tasks"), row=3, col=1)

    fig.update_layout(
        title_text=f"<b>{title}</b>",
        height=950,
        width=1400,
        hovermode="x unified",
        showlegend=True
    )
    fig.show()


# Dashboards

In [29]:
metrics = build_supervisor_dashboard_metrics(
    alerts=alerts,
    cases=cases,
    tasks=tasks,
    period="LAST_30_DAYS",
    value_threshold=5000.0
)

render_supervisor_kpi_cards(metrics)


In [30]:
render_supervisor_operational_charts(metrics)

In [31]:
render_top10_high_value_table(metrics)

In [32]:
start_date, end_date = metrics["date_range"]
trend_df = compute_supervisor_trends(cases, tasks, period="day", start_date=start_date, end_date=end_date)
render_supervisor_trends(trend_df)